# NB02 — Preparación y gestión del dataset de imágenes
**Correspondencia: Semanas 5–8**


## 1. Preparación del entorno


In [ ]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

print("TensorFlow:", tf.__version__)


## 2. Conectar Google Drive

El dataset puede almacenarse en Google Drive para mantener los archivos disponibles entre distintas sesiones de Google Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Definir la ubicación del dataset

La estructura esperada es una carpeta por clase:

```text
dataset/
├── clase_1/
├── clase_2/
├── clase_3/
└── clase_4/
```

Modifique la ruta para apuntar al dataset de su proyecto.


In [ ]:
DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")

print("Ruta:", DATASET_DIR)
print("Existe:", DATASET_DIR.exists())


## 4. Identificar clases y cantidad de imágenes


In [ ]:
extensiones = {".jpg", ".jpeg", ".png", ".bmp"}

clases = sorted([
    carpeta.name
    for carpeta in DATASET_DIR.iterdir()
    if carpeta.is_dir()
])

resumen = []

for clase in clases:
    carpeta = DATASET_DIR / clase
    archivos = [
        f for f in carpeta.iterdir()
        if f.suffix.lower() in extensiones
    ]
    resumen.append({
        "Clase": clase,
        "Cantidad": len(archivos)
    })

df_resumen = pd.DataFrame(resumen)
df_resumen


## 5. Analizar el balance de clases


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(df_resumen["Clase"], df_resumen["Cantidad"])
plt.xlabel("Clase")
plt.ylabel("Cantidad de imágenes")
plt.title("Distribución de imágenes por clase")
plt.xticks(rotation=45)
plt.show()


## 6. Visualizar ejemplos del dataset


In [ ]:
fig, axes = plt.subplots(
    len(clases),
    3,
    figsize=(9, max(3, len(clases) * 3))
)

if len(clases) == 1:
    axes = np.array([axes])

for i, clase in enumerate(clases):
    archivos = [
        f for f in (DATASET_DIR / clase).iterdir()
        if f.suffix.lower() in extensiones
    ]

    seleccion = random.sample(
        archivos,
        min(3, len(archivos))
    )

    for j in range(3):
        ax = axes[i, j]
        ax.axis("off")

        if j < len(seleccion):
            imagen = Image.open(seleccion[j]).convert("RGB")
            ax.imshow(imagen)
            ax.set_title(clase)

plt.tight_layout()
plt.show()


## 7. Revisar dimensiones y formatos


In [ ]:
registros = []

for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue

        try:
            with Image.open(archivo) as img:
                registros.append({
                    "archivo": archivo.name,
                    "clase": clase,
                    "ancho": img.width,
                    "alto": img.height,
                    "formato": img.format,
                    "modo": img.mode
                })
        except Exception:
            pass

df_imagenes = pd.DataFrame(registros)
df_imagenes.head()


In [ ]:
df_imagenes[["ancho", "alto"]].describe()


## 8. Detectar archivos problemáticos

Antes del entrenamiento conviene identificar imágenes dañadas o que no puedan abrirse correctamente.


In [ ]:
archivos_problematicos = []

for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue

        try:
            with Image.open(archivo) as img:
                img.verify()
        except Exception as e:
            archivos_problematicos.append({
                "archivo": str(archivo),
                "error": str(e)
            })

df_problematicos = pd.DataFrame(archivos_problematicos)

print("Archivos problemáticos:", len(df_problematicos))
df_problematicos.head()


## 9. Preparar el dataset con TensorFlow

Definiremos un tamaño común para todas las imágenes y crearemos los conjuntos de entrenamiento y validación.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Clases:", class_names)


## 10. Inspeccionar un batch


In [ ]:
for imagenes, etiquetas in train_ds.take(1):
    print("Forma de las imágenes:", imagenes.shape)
    print("Forma de las etiquetas:", etiquetas.shape)
    print("Rango original:", float(tf.reduce_min(imagenes)), "a", float(tf.reduce_max(imagenes)))


## 11. Normalización

Transformaremos los valores de los píxeles desde el rango 0–255 al rango 0–1.


In [ ]:
normalization_layer = layers.Rescaling(1./255)

train_norm = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_norm = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

for imagenes, etiquetas in train_norm.take(1):
    print(
        "Rango normalizado:",
        float(tf.reduce_min(imagenes)),
        "a",
        float(tf.reduce_max(imagenes))
    )


## 12. División Train / Validation / Test

Para una evaluación final rigurosa se recomienda mantener un conjunto de **test independiente**, que no participe en el entrenamiento ni en la selección del modelo.

Una estructura posible es:

```text
dataset_final/
├── train/
├── validation/
└── test/
```

Cada carpeta debe conservar internamente las mismas clases.

En el proyecto integrador, la dupla deberá documentar claramente la proporción utilizada para cada subconjunto.


## 13. Data Augmentation

El aumento de datos permite generar variaciones de las imágenes de entrenamiento sin modificar las etiquetas.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
])

for imagenes, etiquetas in train_ds.take(1):
    imagen = imagenes[0]

    plt.figure(figsize=(8, 8))

    for i in range(9):
        imagen_aumentada = data_augmentation(
            tf.expand_dims(imagen, 0),
            training=True
        )

        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(
            tf.cast(imagen_aumentada[0], tf.uint8)
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 14. Incorporar Data Augmentation al pipeline


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_aug = train_ds.map(
    lambda x, y: (
        data_augmentation(x, training=True),
        y
    ),
    num_parallel_calls=AUTOTUNE
)

train_aug = train_aug.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


## 15. Verificar etiquetas y clases

El orden de las clases será importante posteriormente, porque deberá mantenerse durante el entrenamiento, la exportación del modelo y la aplicación final.


In [ ]:
for indice, clase in enumerate(class_names):
    print(indice, "→", clase)


## 16. Registrar las características del dataset


In [ ]:
ficha_dataset = {
    "numero_clases": len(class_names),
    "clases": class_names,
    "tamano_imagen": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "normalizacion": "0-1",
    "data_augmentation": [
        "RandomFlip horizontal",
        "RandomRotation 0.10",
        "RandomZoom 0.10"
    ]
}

ficha_dataset


## 17. Actividad

Aplique el notebook al dataset seleccionado para el proyecto integrador y registre:

1. Número total de imágenes.
2. Número de clases.
3. Cantidad de imágenes por clase.
4. Ejemplos representativos de cada categoría.
5. Dimensiones originales de las imágenes.
6. Archivos problemáticos detectados.
7. Estrategia de redimensionamiento y normalización.
8. Distribución definida para train, validation y test.
9. Existencia de desbalance entre clases.
10. Transformaciones de Data Augmentation seleccionadas y su justificación.


## 18. Base para la Evaluación 2

El resultado de este notebook debe permitir disponer de un **dataset preparado, organizado y validado**, listo para ser utilizado posteriormente en la construcción y entrenamiento de la CNN.

**Dataset original → exploración → limpieza → etiquetado → transformación → división → Data Augmentation → dataset preparado**.
